# 第88章 交叉验证与超参数调优

使用交叉验证估计模型波动，并用 GridSearchCV 在训练数据内部选择超参数，最后只在独立测试集评估一次。


## 先解决一个小问题

围绕“交叉验证与超参数调优”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。使用交叉验证估计模型波动，并用 GridSearchCV 在训练数据内部选择超参数，最后只在独立测试集评估一次。


## 这章为什么先学

这是“机器学习”建模主线中的第 88 章，重点放在“交叉验证与超参数调优”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：使用 StratifiedKFold、同时报告均值和标准差、用 GridSearchCV 搜索流水线参数


## 做完要留下什么

完成一份围绕“交叉验证与超参数调优”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 使用 StratifiedKFold
- 同时报告均值和标准差
- 用 GridSearchCV 搜索流水线参数
- 保持最终测试集独立


## 核心概念

- 交叉验证重复利用训练数据估计泛化
- 分层折保持类别比例
- 超参数是拟合前配置而非模型学习参数
- 嵌套选择越多，越需要独立最终测试


## 示例 1：多指标交叉验证

比较逻辑回归在 5 个分层折上的准确率与 ROC-AUC 波动。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

data = load_breast_cancer(as_frame=True); X, y=data.data, data.target
X_train, X_test, y_train, y_test=train_test_split(X, y, stratify=y, test_size=.2, random_state=88)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=88)
scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['accuracy', 'roc_auc'])
print(pd.DataFrame(scores)[['test_accuracy', 'test_roc_auc']].agg(['mean', 'std']).round(4))


## 示例 2：流水线参数搜索

参数名使用步骤名双下划线；搜索只使用训练集。


In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(pipe, {'logisticregression__C':[.01,.1,1,10,100]}, scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)
print('最佳参数:', search.best_params_, ' CV AUC:', round(search.best_score_,4))
print('独立测试 AUC:', round(search.score(X_test, y_test),4))
display(pd.DataFrame(search.cv_results_)[['param_logisticregression__C', 'mean_train_score', 'mean_test_score', 'std_test_score']].round(4))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 调参前已经多次查看测试集
- 只报告最佳均值不报告波动
- 预处理不在 Pipeline 内导致折间泄漏
- 盲目扩大搜索空间造成多重尝试偏差


## 综合练习

1. 同时搜索 penalty='l1'/'l2' 和 C
2. 使用 solver='liblinear'
3. 输出最优参数和测试 AUC

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“同时搜索 penalty='l1'/'l2' 和 C”。
2. **独立完成**：不复制示例代码，完成“使用 solver='liblinear'”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出最优参数和测试 AUC”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
practice_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, solver='liblinear'))
practice_search = GridSearchCV(practice_pipe, {'logisticregression__C':[.01,.1,1,10], 'logisticregression__penalty':['l1', 'l2']}, scoring='roc_auc', cv=cv, n_jobs=-1)
practice_search.fit(X_train, y_train)
practice_test_score = practice_search.score(X_test, y_test)
print(practice_search.best_params_, round(practice_test_score,4))

# 自检
assert 0 <= practice_test_score <= 1
assert 'logisticregression__penalty' in practice_search.best_params_


## 本章小结

使用交叉验证估计模型波动，并用 GridSearchCV 在训练数据内部选择超参数，最后只在独立测试集评估一次。


### 你已经掌握

- 使用 StratifiedKFold
- 同时报告均值和标准差
- 用 GridSearchCV 搜索流水线参数
- 保持最终测试集独立


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 多指标交叉验证 | 比较逻辑回归在 5 个分层折上的准确率与 ROC-AUC 波动。 | `pd.DataFrame()`、`.agg()`、`.round()` |
| 流水线参数搜索 | 参数名使用步骤名双下划线；搜索只使用训练集。 | `search.fit()`、`search.score()`、`pd.DataFrame()`、`.round()` |


### 需要注意

- 调参前已经多次查看测试集
- 只报告最佳均值不报告波动
- 预处理不在 Pipeline 内导致折间泄漏
- 盲目扩大搜索空间造成多重尝试偏差


### 完成检查

- [ ] 能够使用 StratifiedKFold
- [ ] 能够同时报告均值和标准差
- [ ] 能够用 GridSearchCV 搜索流水线参数
- [ ] 能够保持最终测试集独立


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
